In [1]:
import os
import sys

# Setup path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.chdir(project_root)
print(f'✅ Working directory: {os.getcwd()}')

✅ Working directory: c:\Users\lucas\workspace\github\fase-5


# 📖 Análise de Continuidade Estudantil — Datathon Passos Mágicos

## 🎯 Objetivo

Exploração temporal do dataset PEDE 2024 com foco em **continuidade e retenção de alunos** entre 2020-2022:
- Separar dados em cohorts anuais (2020, 2021, 2022)
- Analisar padrões de permanência e abandono
- Identificar alunos ingressados vs veteranos
- Informar estratégias de acompanhamento pedagógico

---

## 🧠 Lógica de Negócio

### 1. Questões Chave de Negócio

| Pergunta | Impacto | Métrica |
|----------|--------|---------|
| **Quantos alunos continuam de um year para outro?** | Retenção | Taxa de continuidade (%) |
| **Quantos alunos eram novos em 2022?** | Growth / Churn | Novos ÷ Total em 2022 |
| **Qual é a taxa de dropout entre years?** | Risk | 1 - Taxa Continuidade |
| **Quem são os alunos de risco?** | Intervention | Alunos com padrão de absenteísmo |

### 2. Arquitetura de Análise

```
Dataset Bruto (PEDE 2024)
    ↓
📊 filter_columns(df, ['_20', '_21', '_22'])
    ↓
🧹 cleaning_dataset(df)  → Remove linhas vazias
    ↓
📅 create_annual_datasets(df)  → {2020: df_20, 2021: df_21, 2022: df_22}
    ↓
🔗 analyze_student_continuity(df_20, df_21, df_22)
    ↓
📈 Resultados:
   - Continuidade 2020→2021: X de Y alunos
   - Continuidade 2021→2022: A de B alunos
   - Novos em 2022: C alunos
   - Dropout rate: Z%
```

### 3. Funções Importadas (sem inline code)

Todas as funções estão em **`scripts/datathon_cleaning.py`** com testes em **`tests/scripts/test_datathon_cleaning.py`**:

#### 1️⃣ `filter_columns(df, filters: List[str]) → DataFrame`
- **Input**: DataFrame + lista de padrões ("_20", "_21", "_22")
- **Output**: DataFrame com apenas colunas que contêm esses padrões
- **Uso**: Isolar dados por year

#### 2️⃣ `cleaning_dataset(df) → DataFrame`
- **Input**: DataFrame (potencialmente com linhas vazias)
- **Output**: DataFrame sem linhas completamente nulas (exceto NOME)
- **Uso**: Remover ruído de registros vazios

#### 3️⃣ `create_annual_datasets(df) → Dict[int, DataFrame]`
- **Input**: DataFrame consolidado com colunas de múltiplos anos
- **Output**: Dicionário {2020: df_2020, 2021: df_2021, 2022: df_2022}
- **Tratamento**: Remove sufixos de ano das colunas automaticamente
- **Uso**: Preparar dados para análise temporal

#### 4️⃣ `analyze_student_continuity(df_2020, df_2021, df_2022) → Dict[str, Any]`
- **Input**: 3 DataFrames anuais
- **Output**: Dicionário com métricas de continuidade:
  ```python
  {
    "continuidade_2020_2021": int,  # Alunos em ambos years
    "taxa_2020_2021": float,        # Percentual (0-100)
    "continuidade_2021_2022": int,
    "taxa_2021_2022": float,
    "novos_2022": int,              # Ingressados apenas em 2022
    "alunos_2020": int,
    "alunos_2021": int,
    "alunos_2022": int,
    "set_2020": set,                # Nomes únicos de alunos
    "set_2021": set,
    "set_2022": set,
  }
  ```

### 4. Garantias de Qualidade ✅

| Aspecto | Garantia |
|--------|----------|
| **Whitespace** | Nomes são .strip()'d antes de comparação |
| **Duplicatas** | Usa .unique() para eliminar duplicatas |
| **Type Safety** | Type hints em todas as assinaturas |
| **Edge Cases** | Trata DataFrames vazios, NOME missing, etc |
| **Testing** | 38 testes unitários em `test_datathon_cleaning.py` |

### 5. Próximos Passos (Análise)

Após executar este notebook, você terá:

✅ **Datasets separados** por year (df_2020, df_2021, df_2022)  
✅ **Métricas de continuidade** (taxa de permanência entre years)  
✅ **Identificação de novos alunos** em 2022  
✅ **Insight para estratégia**: Direcionar acompanhamento aos que podem evadir  

---

## 🔗 Referências

- **Código**: `scripts/datathon_cleaning.py` (4 funções, ~160 linhas)
- **Testes**: `tests/scripts/test_datathon_cleaning.py` (38 testes, ~400 linhas)
- **Documentação**: Google-style docstrings em cada função
- **Type Hints**: Python 3.11+ annotations em todos os parâmetros/returns

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from scripts.datathon_cleaning import (
    analyze_student_continuity,
    cleaning_dataset,
    create_annual_datasets,
    filter_columns,
)

# Configurações
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Imports concluídos - funções importadas de scripts/datathon_cleaning')

✅ Imports concluídos - funções importadas de scripts/datathon_cleaning


## 1. Carregamento do Dataset

In [ ]:
# Carregamento do dataset bruto
file_path = './PEDE_PASSOS_DATASET_FIAP.csv'
df = pd.read_csv(file_path, delimiter=';')

print(f'📊 Dataset: {df.shape[0]} linhas × {df.shape[1]} colunas')
display(df.head())

## 3. Datasets por Ano

In [ ]:
# Usar função create_annual_datasets para criar todos os datasets de uma vez
datasets = create_annual_datasets(df)

df_2020 = datasets[2020]
df_2021 = datasets[2021]
df_2022 = datasets[2022]

print(f'✅ 2020: {len(df_2020)} alunos')
print(f'✅ 2021: {len(df_2021)} alunos')
print(f'✅ 2022: {len(df_2022)} alunos')

## 4. Análise de Continuidade

In [ ]:
# Usar função analyze_student_continuity para análise completa
result = analyze_student_continuity(df_2020, df_2021, df_2022)

print('📊 Continuidade entre Anos:')
print(f'   2020 → 2021: {result["continuidade_2020_2021"]}/{result["alunos_2020"]} ({result["taxa_2020_2021"]:.1f}%)')
print(f'   2021 → 2022: {result["continuidade_2021_2022"]}/{result["alunos_2021"]} ({result["taxa_2021_2022"]:.1f}%)')
print(f'   Novos em 2022: {result["novos_2022"]}')

## 5. Ideias para Pesquisa Futura

- 🎯 **Recomendação de Bolsas**: Priorizar alunos com maior risco de abandono
- 📊 **Análise de Dropout**: Identificar fatores de evasão
- 🔍 **Outliers**: Alunos com melhor desempenho como estudo de caso
- 🔮 **Predição Futura**: Usar dados 2020-2021 para prever 2022